# scNestedWGCNA — tutorial

This notebook runs the scNestedWGCNA pipeline (`src/`) on a SEACells metacell
expression table, step by step:

1. Load summed SEACells metacell counts and normalize them (cell-type-aware poscounts)
2. Load a gene regulatory network (GRN), required for the `knowledge_score` objective
3. Optimize UMAP embedding parameters
4. Find CGM (Coarse-Grained Module) parameters and run the final CGM clustering
5. Visualize the CGMs in 3D
6. Extract FGMs (Fine-Grained Modules) for one CGM, step by step
7. Compare unnormalized vs. regression-normalized gene embeddings for that CGM
8. Extract FGMs for **all** CGMs, with unnormalized and normalized embeddings per gene
9. Run and save the full FGM pipeline (all CGMs) to GMT files
10. Inspect the saved GMT output
11. The equivalent command-line invocation, for unattended runs

**Pipeline in brief**

- **CGMs** are gene clusters found on the whole gene set: UMAP of the gene–gene
  dissimilarity, then HDBSCAN. The sweep over `min_samples` and `min_cluster_size`
  reuses one UMAP embedding and one HDBSCAN tree per `min_samples` value, and
  re-condenses that tree at each `min_cluster_size`.
- **FGMs** are gene clusters inside each CGM. Each CGM's genes are normalized
  (log-space regression on their shared confounder signal), clustered the same
  way, and then merged at a coarser size.
- **Merging** uses the same tree at a coarser `min_cluster_size * merge_size_multiplier`.
  That coarse cut only votes: each fine FGM goes as a whole to the coarse cluster
  that most of its genes fall into. Merged FGMs are therefore unions of fine FGMs,
  and there are never more merged FGMs than fine ones.
- **Noise** is one label, `module_-1`. It collects genes removed by the optional
  pre-filter, CGM noise, FGM noise (kME test on the merged labels, or hard HDBSCAN
  noise with `noise_method="hdbscan"`), and the non-core genes of the `*_core` columns.
- **Outputs** are GMT files: fine and merged modules, each as raw (with noise) and
  filtered (dense core only).

**Kernel**: run this notebook with the `sc_analysis` conda environment
(`conda activate sc_analysis`; the registered Jupyter kernel is `python3`
under that env).

**Gene network**: `knowledge_score` scores a clustering by the odds ratio of
co-clustered gene pairs landing on edges of a reference network. A
`gene_network` DataFrame (or a `gene_pathway_dict`) is therefore required.

## 0. Setup

`src` is a package at the project root (`src/__init__.py` exists), and
its modules use absolute imports like `from src.utils import ...`. So the
**project root must be on `sys.path`**. This is the same as running the
pipeline modules from the project root.

In [1]:
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd
import optuna

# Project root = parent of notebooks/ (run with notebooks/ as the working directory)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
DATA_ID = "1140"
print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Results dir:", RESULTS_DIR)


Project root: /Users/artux/Documents/scNestedWGCNA
Data dir: /Users/artux/Documents/scNestedWGCNA/data


## 1. Load and normalize the SEACells metacell counts

`run_seacells.py` saves **raw summed counts** (metacells x genes).
The pipeline expects already normalized, linear-scale input, so normalization
is a separate step, `src/normalization.py`: DESeq2-style *poscounts* size factors
are estimated within each group (here the cell type, taken from the metacell
names: `pop2_MC012` -> `pop2`), the groups are then put on a common scale, and
the result is rescaled so the median metacell total is 1e4. The same step is
available on the command line (`python run_metacell_normalization.py`, or
`--out-norm-filename` in `run_seacells.py`).

We use `data/metacells/seacells_sum_1140.csv`: 98 metacells x 1540 genes
(7 simulated populations x 14 metacells) from the scMultiSim "1140" dataset.
It is paired with a GRN reference for the same simulation
(`data/grns/grn_multisim_1140.csv`), needed in the next step. As in the
pipeline's loader, genes that are zero in every metacell are dropped.

In [3]:
from src.normalization import groups_from_metacell_names, normalize_metacell_counts

counts_path = os.path.join(DATA_DIR, "metacells", f"seacells_sum_{DATA_ID}.csv")
counts_df = pd.read_csv(counts_path, index_col=0)
groups = groups_from_metacell_names(counts_df.index)

expr_df, size_factors_df, group_scale = normalize_metacell_counts(
    counts_df, groups, min_mean=1.0, target_total=1e4
)
expr_df = expr_df.loc[:, (expr_df != 0).any(axis=0)]  # what load_and_filter_data() does

print("Expression matrix:", expr_df.shape, "(metacells x genes)")
print("Group scales:", group_scale.round(2).to_dict())
expr_df.iloc[:5, :5]

Expression matrix: (98, 1540) (metacells x genes)
Group scales: {'pop2': 1.21, 'pop3': 1.27, 'pop4': 1.05, 'pop5': 0.99, 'pop6': 0.99, 'pop7': 0.76, 'pop8': 0.84}


,gene1,gene2,gene3,gene4,gene5
pop2_MC012,1.041359,3.272842,4.254695,2.826545,8.806920
pop2_MC007,1.270419,4.234731,2.286755,2.202060,3.641869
pop2_MC004,1.440623,6.554836,3.817652,1.008436,5.258275
pop2_MC011,1.090426,3.271277,3.587852,1.899451,4.643103
pop2_MC005,1.435943,3.988730,6.169236,2.180506,7.020165


### 1b. Optional: drop genes with no co-expression partner (before CGM clustering)

`topk_coexpression_filter()` runs **before** UMAP optimization and CGM
clustering (CLI: `--prefilter-genes`, with `--prefilter-k/-n-perm/-fdr`;
the CLI default is `none`, i.e. off). It flags genes whose strongest `k` Spearman
correlations with the other genes are not larger than expected from permuted data
(Benjamini–Hochberg at `fdr`).

Set `APPLY_PREFILTER = False` to skip it (then all genes are clustered, as in a default CLI run). When on:
- all later steps use the filtered matrix, and the CGM `min_clust_size` fractions are relative to the filtered gene count;
- the removed genes are kept in `prefilter_noise_genes` and passed to `extract_all_fgms(extra_noise_genes=...)` (Step 8): they end up
  in the noise module `module_-1` (all label columns; NaN embeddings). The fine `module` column keeps the origin of every other gene.
  The CLI does the same: the removed genes are written to the raw GMTs as `module_-1`.

In [4]:
from src.utils import topk_coexpression_filter

APPLY_PREFILTER = True   # CLI equivalent: --prefilter-genes (default: off)

prefilter_noise_genes = expr_df.columns[:0]   # genes removed here; empty if skipped
if APPLY_PREFILTER:
    t0 = time.time()
    prefilter = topk_coexpression_filter(expr_df, k=5, n_perm=500, fdr=0.01)   # CLI defaults
    is_noise = prefilter.is_noise.to_numpy()
    prefilter_noise_genes = expr_df.columns[is_noise]
    expr_df = expr_df.loc[:, ~is_noise]
    print(f"[{time.time()-t0:.1f}s] removed {is_noise.sum()} of {len(is_noise)} genes "
          f"({is_noise.mean():.1%}); {expr_df.shape[1]} remain -> expression matrix {expr_df.shape}")
else:
    print("Pre-filter skipped; expression matrix", expr_df.shape)

[68.2s] removed 56 of 1540 genes (3.6%); 1484 remain -> expression matrix (98, 1484)


## 2. Load the gene regulatory network (required for `knowledge_score`)

`knowledge_score` scores a clustering by the odds ratio of co-clustered gene
pairs (same cluster) landing on edges of a reference knowledge graph, versus
not. The reference graph can come from a `gene_network` DataFrame (columns read
positionally as `target, regulator, ...`) or from a `gene_pathway_dict`. Here we
use the simulation's ground-truth GRN as the reference. In a real analysis this
would be a published regulatory network, such as DoRothEA or CollecTRI.

In [5]:
from run_nested_wgcna import load_gene_network


if DATA_ID in ["1140", "d1k"]:
    gene_network_path = os.path.join(DATA_DIR, "grns", "grn_multisim_1140.csv")
elif DATA_ID == "2150":
    gene_network_path = os.path.join(DATA_DIR, "grns", "grn_multisim_2150.csv")
else:
    raise ValueError(f"{DATA_ID = } is wrong value.")


gene_network = load_gene_network(gene_network_path)

print("Gene network edges:", gene_network.shape)
gene_network.head()

Gene network edges: (1432, 3)


,target,regulator,effect
0,gene1,gene684,4.057811
1,gene1,gene803,2.312057
2,gene3,gene684,4.469269
3,gene3,gene803,2.267506
4,gene4,gene684,2.231499


## 3. UMAP embedding parameters

`optimize_umap_params()` runs an Optuna TPE search over `n_neighbors`,
`n_components` and `min_dist`. Each candidate is scored by a composite of
trustworthiness (40%), density preservation (40%), label consistency across
5 random seeds (15%), and DBCV (5%).

The CLI default is `n_trials=50`. This tutorial uses `n_trials=8` to keep the
runtime short (about 2 minutes). For a full run, use the default
(`--n-trials-umap 50` on the CLI).

In [6]:
from src.parameter_optimization import optimize_umap_params

t0 = time.time()
best_umap_params = optimize_umap_params(
    expr_df,
    min_clust_size=30,   # only used to score candidate embeddings during the search
    min_samples=3,
    n_trials=10,          # CLI default: 50
    n_startup_trials=3,
    n_warmup_steps=3,
)
print(f"\n[{time.time()-t0:.1f}s] best_umap_params = {best_umap_params}")


Clustering with min_clust_size = 30 and min_samples = 3.
Clustering done.
Clustering with min_clust_size = 30 and min_samples = 3.
Clustering done.
Clustering with min_clust_size = 30 and min_samples = 3.
Clustering done.
Clustering with min_clust_size = 30 and min_samples = 3.
Clustering done.
Clustering with min_clust_size = 30 and min_samples = 3.
Clustering done.
Clustering with min_clust_size = 30 and min_samples = 3.
Clustering done.
Clustering with min_clust_size = 30 and min_samples = 3.
Clustering done.
Clustering with min_clust_size = 30 and min_samples = 3.
Clustering done.
Clustering with min_clust_size = 30 and min_samples = 3.
Clustering done.
Clustering with min_clust_size = 30 and min_samples = 3.
Clustering done.
Clustering with min_clust_size = 30 and min_samples = 3.
Clustering done.
Clustering with min_clust_size = 30 and min_samples = 3.
Clustering done.
Clustering with min_clust_size = 30 and min_samples = 3.
Clustering done.
Clustering with min_clust_size = 30 an

## 4. CGM parameters and final CGM clustering

CGM `min_cluster_size` is searched as **fractions of total gene count**
(CLI defaults: `--cgm-min-clust-frac-start 0.03 --cgm-min-clust-frac-stop
0.10 --cgm-min-clust-frac-step 0.02`). `frac_range()` builds that grid, and
`get_cgm_params()` in `run_nested_wgcna.py` converts the fractions to gene counts.

`optimize_cgm_clustering_params()` fits one UMAP embedding (using the
`best_umap_params` above) and, for each candidate `min_samples`, one HDBSCAN
tree. It then re-condenses that tree at every candidate `min_cluster_size`.
The sweep takes well under a minute.

In [7]:
from run_nested_wgcna import frac_range
from src.parameter_optimization import optimize_cgm_clustering_params

cgm_min_clust_frac_lst = frac_range(start=0.15, stop=0.20, step=0.01)
min_clust_size_lst = sorted(
    {max(5, round(frac * expr_df.shape[1])) for frac in cgm_min_clust_frac_lst}
)
print("CGM min_cluster_size candidates:", min_clust_size_lst)

t0 = time.time()
best_cgm_params = optimize_cgm_clustering_params(
    expr_df,
    best_umap_params=best_umap_params,
    gene_network=gene_network,
    min_clust_size_lst=min_clust_size_lst,
)
print(f"\n[{time.time()-t0:.1f}s] best_cgm_params = {best_cgm_params}")

CGM min_cluster_size candidates: [223, 237, 252, 267, 282, 297]

----- CGM grid search (cheap re-condensing sweep, src) -----
 min_clust_size  min_samples  knowledge_score
            237            5         2.554886
            223            5         2.554886
            282            5         2.554886
            297            5         2.554886
            267            5         2.554886
            252            5         2.554886
            252            4         2.522065
            267            4         2.522065
            237            4         2.522065
            223            4         2.522065
----- Best params: {'min_clust_size': 223, 'min_samples': 5, 'n_neighbors': 10, 'n_components': 8, 'min_dist': 0.06011150117432088}
----- Best score: 2.554885778986176


[6.3s] best_cgm_params = {'min_clust_size': 223, 'min_samples': 5, 'n_neighbors': 10, 'n_components': 8, 'min_dist': 0.06011150117432088}


In [78]:
from src.utils import get_cgms

dissim, embedding, cgm_df, module_counts, clusterer = get_cgms(
    expr_df,
    min_clust_size=best_cgm_params["min_clust_size"],
    min_samples=best_cgm_params["min_samples"],
    n_neighbors=best_cgm_params["n_neighbors"],
    n_components=best_cgm_params["n_components"],
    min_dist=best_cgm_params["min_dist"],
    noise_method="hdbscan",  # every gene -> its soft-argmax CGM; "hdbscan" marks native noise as module_-1
)

print(f"Number of CGMs (excluding noise): {(module_counts.module != 'module_-1').sum()}")
module_counts


Clustering with min_clust_size = 223 and min_samples = 5.
Clustering done.
Number of CGMs (excluding noise): 2


,module,count
0,module_1,685
1,module_0,463
2,module_-1,336


`cgm_df` has one row per gene, indexed by gene name, with two columns:
- `module`: the CGM assignment. With `noise_method="none"` every gene gets a cluster; with `"hdbscan"`, HDBSCAN's noise genes become `module_-1`.
- `core_module`: the dense-core subset after graph-degeneracy erosion (`errode_clusters()`). Non-core genes are `module_-1`.

In [ ]:
cgm_df.head()

,module,core_module
gene1,module_3,module_3
gene2,module_5,module_5
gene3,module_3,module_3
gene4,module_1,module_1
gene5,module_3,module_3


## 5. Visualize the CGMs in 3D

`plot_3d()` renders the first 3 components of a UMAP embedding as an
interactive Plotly scatter, coloured by module. Genes are coloured by
`cgm_df["module"]`, with noise shown as `module_-1`.

In [79]:
from src.utils import plot_3d

plot_3d(embedding, cgm_df["module"].values, title="CGM embedding")


## 6. FGMs for all CGMs, with unnormalized and normalized embeddings

Step 6 handled one CGM. `extract_all_fgms()` runs the same `get_fgms_from_cgm()`
for **every** non-noise CGM (largest first). With `return_embeddings=True` it also
keeps, for every gene, the two embeddings that Step 7 plotted for one CGM: a UMAP fit
on the CGM's unnormalized expression (`raw_umap1..d`) and one on the
regression-normalized expression that was clustered (`norm_umap1..d`). This costs
one extra UMAP fit per CGM; the CLI leaves it off.

**Each CGM is its own UMAP fit**, so coordinates are comparable *within* a CGM
(raw vs. normalized, FGM colours) but **not across CGMs**. Plot one CGM at a time.

`gene_table` has one row per gene: `cgm` / `cgm_core` (Step 4), the FGM columns
(`module`, `module_core`, `merged_module`, `merged_module_core`, `kme`, `kme_qval`)
and the embedding columns. Genes of a CGM that fell back to an unrefined block
(see Step 9) have NaN embeddings. Noise genes (pre-filter, CGM noise, kME, and the
non-core genes of the `*_core` columns) all carry the label `module_-1`. Pre-filter
and CGM-noise genes have NaN embeddings, because they were in no CGM's UMAP. kME
noise genes keep theirs.

**Runtime**: this is the full per-CGM FGM stage for all CGMs, run sequentially
(`n_jobs=1`). On this data (98 metacells, 1540 genes, 7 CGMs) it took about 18 s
on the author's machine (one run). Larger data takes longer.

In [99]:
from run_nested_wgcna import extract_all_fgms

fgm_min_clust_frac_lst = frac_range(start=0.01, stop=0.05, step=0.01)   # CLI default
merge_size_multiplier_lst = frac_range(start=1.5, stop=3.0, step=0.5)   # CLI default

t0 = time.time()
fgm_df = extract_all_fgms(
    expr_df,
    cgm_df,
    best_umap_params,
    optimize_clust_params=True,
    fgm_min_clust_frac_lst=fgm_min_clust_frac_lst,
    merge_size_multiplier_lst=merge_size_multiplier_lst,
    gene_network=gene_network,
    n_jobs=1,
    noise_method="kme",   # FGM noise: "kme" (soft labels + kME test) or "hdbscan" (hard labels, native -1)
    # fgm_noise_kwargs={"kme_fdr": 0.05, "kme_min": 0.3},   # kME settings (noise_method="kme")
    return_embeddings=True,
    extra_noise_genes=prefilter_noise_genes,   # Step 1b removals -> module_-1
)
print(f"\n[{time.time()-t0:.1f}s] FGMs extracted for all CGMs")

Processing CGM: module_1
Normalization parameter selection (Tier 1: INGS bootstrap stability).

----- Confounder Tier 1 shortlist (bootstrap stability) -----
corr_thr=0.75, n_INGS=23, ings_fraction=0.0336, stability=0.2932
corr_thr=0.8, n_INGS=10, ings_fraction=0.0146, stability=0.2537

Submodules identification (Tier 2: scoring 2 corr_thr x 1 k candidates by downstream knowledge_score).

----- FGM grid search (cheap re-condensing sweep, src) -----
 min_clust_size  min_samples  merge_size_multiplier  coarse_size  knowledge_score
              7            1                    1.5           10       146.403388
              7            3                    2.5           18        72.724319
              7            3                    2.0           14        72.724319
              7            3                    1.5           10        72.724319
              7            3                    3.0           21        72.724319
             14            3                    1.5    

In [100]:
gene_table = (
    cgm_df[["module", "core_module"]]
    .rename(columns={"module": "cgm", "core_module": "cgm_core"})
    .reindex(fgm_df.index)
    .fillna("module_-1")   # pre-filtered genes are not in cgm_df: CGM-level noise
    .join(fgm_df)
)
embedding_cols = [c for c in gene_table.columns if c.startswith(("raw_umap", "norm_umap"))]
print(f"gene_table: {gene_table.shape[0]} genes x {gene_table.shape[1]} columns "
      f"({len(embedding_cols)} embedding columns)")
print("\nFine FGMs per CGM:")
print(gene_table.groupby("cgm")["module"].nunique().rename("n_fine_fgms").to_frame()
      .join(gene_table.cgm.value_counts().rename("n_genes")))

# One noise module for every stage (pre-filter, CGM noise, kME, core erosion). Fine `module` keeps the origin.
print("\nNoise genes (module_-1) per column:",
      (gene_table[["cgm", "merged_module", "merged_module_core", "module_core"]] == "module_-1").sum().to_dict())
gene_table[["cgm", "module", "merged_module"] + embedding_cols[:2]].head()

gene_table: 1540 genes x 24 columns (16 embedding columns)

Fine FGMs per CGM:
           n_fine_fgms  n_genes
cgm                            
module_-1            1      392
module_0             2      463
module_1             2      685

Noise genes (module_-1) per column: {'cgm': 392, 'merged_module': 498, 'merged_module_core': 597, 'module_core': 529}


,cgm,module,merged_module,raw_umap1,raw_umap2
gene2,module_1,module_1_0,module_1_0,5.327422,3.425544
gene10,module_1,module_1_0,module_1_0,5.590957,3.454895
gene11,module_1,module_1_0,module_1_0,5.453587,3.422957
gene14,module_1,module_1_0,module_1_0,6.107445,4.385189
gene22,module_1,module_1_1,module_1_4,0.230422,1.502507


In [101]:
gene_table["module"].value_counts()

module
module_1_1    484
module_-1     392
module_0_0    239
module_0_1    224
module_1_0    201
Name: count, dtype: int64

In [102]:
gene_table["merged_module"].value_counts()

merged_module
module_-1      498
module_0_0     211
module_1_0     200
module_1_4     180
module_0_1     164
module_1_10     54
module_0_3      46
module_1_8      40
module_1_3      34
module_1_7      30
module_1_9      23
module_1_6      18
module_0_2      18
module_1_2      10
module_1_1       9
module_1_5       5
Name: count, dtype: int64

In [87]:
def plot_cgm_embeddings(cgm_name, color_col="module"):
    """Unnormalized and normalized 3D UMAP of one CGM's genes, coloured by `color_col`."""
    sub = gene_table[gene_table.cgm == cgm_name]
    for kind, title in (("raw", "unnormalized"), ("norm", "regression-normalized")):
        plot_3d(
            sub[[f"{kind}_umap{i}" for i in (1, 2, 3)]].to_numpy(),
            sub[color_col].values,
            title=f"{cgm_name}: {title} expression",
        )

# One CGM at a time (embeddings of different CGMs share no coordinates):
plot_cgm_embeddings("module_1")

# Every CGM (many interactive figures):
# for name in gene_table.cgm.unique():
#     plot_cgm_embeddings(name)

In [88]:
from run_nested_wgcna import save_modules_to_gmt


output_file_name = f"fgm_filt_{DATA_ID}_multisim_cgm_hdbscan_fgm_kme"

specs = [
    ("module", output_file_name, "Fine raw modules"),
    ("merged_module", f"{output_file_name}_merged", "Merged raw modules"),
]
outputs = []
for column, stem, label in specs:
    d = {
        key: fgm_df[fgm_df[column] == key].index.to_list()
        for key in fgm_df[column].unique()
    }

    outputs.append((d, os.path.join(RESULTS_DIR, "modules", f"{stem}.gmt"), label))

for d, path, _ in outputs:
    save_modules_to_gmt(d, path)

print(f"\nSaved results to:")
for _, path, label in outputs:
    print(f"  {label}: {path}")


Saved results to:
  Fine raw modules: /home/adakhnovets/scRNA_PhD_project/data/fgm_filt_1140_multisim_cgm_hdbscan_fgm_kme.gmt
  Merged raw modules: /home/adakhnovets/scRNA_PhD_project/data/fgm_filt_1140_multisim_cgm_hdbscan_fgm_kme_merged.gmt


## 7. Run and save the full FGM pipeline (all CGMs)

`extract_and_save_fgms()` runs Step 6's `get_fgms_from_cgm()` for **every**
non-noise CGM (through the picklable `_process_cgm()` worker, dispatched via
`joblib.Parallel` when `n_jobs != 1`) and writes **four** GMT files:

| file | column used | noise included? |
|---|---|---|
| `{prefix}_raw.gmt` | `module` (fine) | yes |
| `{prefix}_filtered.gmt` | `module_core` (fine, dense core) | no |
| `{prefix}_raw_merged.gmt` | `merged_module` | yes |
| `{prefix}_filtered_merged.gmt` | `merged_module_core` | no |

A CGM whose FGM extraction raises is kept as one unrefined block, and a
`{prefix}_raw_failed_cgms.csv` (CGM, n_genes, error) is written next to the
GMTs. With `--fgm-noise-method kme` (default), a `{prefix}_fgm_kme.csv` with
per-gene `kme` / `kme_qval` is written as well (pass `diagnostics_path=` when
calling `extract_and_save_fgms()` directly).

**Runtime note**: Step 8 ran all 7 CGMs here in about 18 s. On larger datasets the
FGM stage is the expensive part of the pipeline. For a real analysis, run it as a
background or unattended job (see Step 11). The cell below is commented out;
uncomment it to run it.

In [18]:
from run_nested_wgcna import extract_and_save_fgms

# fgm_min_clust_frac_lst / merge_size_multiplier_lst: defined in Step 8
output_dir = os.path.join(RESULTS_DIR, "modules")
output_file_names = ["tutorial_1140_raw", "tutorial_1140_filtered"]

# Uncomment to actually run the full sweep and write the GMTs:
#
# t0 = time.time()
# fgm_df = extract_and_save_fgms(
#     expr_df,
#     cgm_df,
#     best_umap_params,
#     fgm_min_clust_frac_lst,
#     merge_size_multiplier_lst,
#     optimize_clust_params=True,
#     gene_network=gene_network,
#     output_dir=output_dir,
#     output_file_names=output_file_names,
#     n_jobs=-1,                 # use all available cores, one CGM per worker
# )
# print(f"[{time.time()-t0:.1f}s] done")

print("Would write to:", output_dir)
print("Output files:", [f"{n}.gmt" for n in output_file_names] +
      [f"{n}_merged.gmt" for n in output_file_names])

Would write to: /Users/artux/Documents/scNestedWGCNA/data/modules
Output files: ['tutorial_1140_raw.gmt', 'tutorial_1140_filtered.gmt', 'tutorial_1140_raw_merged.gmt', 'tutorial_1140_filtered_merged.gmt']


## 8. Inspect saved GMT output

GMT files live in `results/modules/` (pipeline outputs) and `data/gene_modules/`
(ground-truth and benchmark modules). Each line is
`module_name<TAB>description<TAB>gene1<TAB>gene2<TAB>...`.

In [19]:
def load_gmt(path):
    modules = {}
    with open(path) as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            name = parts[0]
            genes = [g for g in parts[2:] if g]
            modules[name] = genes
    return modules

# Example: the ground-truth modules of this simulation (data/gene_modules/); once
# Step 9 has been run, point this at results/modules/tutorial_1140_filtered_merged.gmt
example_path = os.path.join(DATA_DIR, "gene_modules", "gtmodule_1140_multisim.gmt")
if os.path.exists(example_path):
    modules = load_gmt(example_path)
    print(f"{example_path}\n  {len(modules)} modules")
    for name, genes in list(modules.items())[:5]:
        print(f"  {name}: {len(genes)} genes")
else:
    print(f"{example_path} not found — run Step 9 (or the CLI, Step 11) to produce GMT output.")

/Users/artux/Documents/scNestedWGCNA/data/gene_mods/gtmodule_1140_multisim_upd.gmt
  7 modules
  mod1: 276 genes
  mod2: 185 genes
  mod3: 243 genes
  mod4: 174 genes
  mod5: 198 genes


## 9. Equivalent command-line run

Everything above can also be run unattended: normalize the summed counts, then
run the pipeline. This is the recommended way to run a full sweep, since it can
take a long time and does not need a live notebook kernel. Both commands must be
run from the **project root** (the scripts import the library as `from src.*`).

```bash
conda activate sc_analysis
cd /path/to/scNestedWGCNA

# 1) cell-type-aware poscounts normalization of the summed counts
#    (groups default to the metacell name before the last "_")
python run_metacell_normalization.py \
  --input data/metacells/seacells_sum_1140.csv \
  --output data/metacells/seacells_norm_1140.csv

# 2) nested WGCNA on the normalized table
python run_nested_wgcna.py \
  --input-file metacells/seacells_norm_1140.csv \
  --output-prefix fgm_results_1140 \
  --gene-network-file data/grns/grn_multisim_1140.csv \
  --optimize-umap --optimize-cgm --optimize-fgm \
  --n-trials-umap 50 \
  --n-jobs -1
```

Outputs go to `results/modules/` (change with `--output-dir`).

Notes:
- `--gene-network-file` is only required with `--optimize-cgm` / `--optimize-fgm`. `knowledge_score` is the only supported objective.
- `--optimize-umap`, `--optimize-cgm` and `--optimize-fgm` are opt-in flags (off by default). Without them the `--umap-*` values (default 15/10/0.1) and `--cgm-*` values (`min_clust_size` 50, `min_samples` 3) are used.
- Noise handling: `--cgm-noise-filter {none,hdbscan}` (default `none`), `--prefilter-genes` (off by default; tuned by `--prefilter-k/-n-perm/-fdr`) and `--fgm-noise-method {hdbscan,kme}` (default `kme`, with `--kme-fdr` / `--kme-min`).
- `--n-jobs -1` parallelizes FGM extraction across CGMs (one CGM per worker), and each worker's HDBSCAN thread count is throttled accordingly.
- Run `python run_nested_wgcna.py ...` from the project root (the CLI scripts live there; the library code is in `src/`).
- Summed counts come from `run_seacells.py` (`seacells_env`). It can also write the normalized table directly with `--out-norm-filename`.